# <font color = 'red'> Dependencias

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio, count_by_category, proportions_by_category, counts_and_proportions_by_category)
from visualization_tools import plot_interactive_chart, plot_categorical_proportions
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> Carga de Datos

In [2]:
df = pd.read_csv(get_data_path("preprocessed_data.csv"))

df = df.filter(regex = 'Occupation|Credit_Mix|Credit_Score').drop('Binary_Credit_Score', axis = 1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 17 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   Credit_Mix                100000 non-null  object
 1   Credit_Score              100000 non-null  int64 
 2   Occupation_Accountant     100000 non-null  int64 
 3   Occupation_Architect      100000 non-null  int64 
 4   Occupation_Developer      100000 non-null  int64 
 5   Occupation_Doctor         100000 non-null  int64 
 6   Occupation_Engineer       100000 non-null  int64 
 7   Occupation_Entrepreneur   100000 non-null  int64 
 8   Occupation_Journalist     100000 non-null  int64 
 9   Occupation_Lawyer         100000 non-null  int64 
 10  Occupation_Manager        100000 non-null  int64 
 11  Occupation_Mechanic       100000 non-null  int64 
 12  Occupation_Media_Manager  100000 non-null  int64 
 13  Occupation_Musician       100000 non-null  int64 
 14  Occup

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
Duplicate rows found: 99955


# <font color = 'red'> Análisis

In [5]:
occupation_cols = [col for col in df.columns if col.startswith("Occupation_")]

results = counts_and_proportions_by_category(df, occupation_cols, "Credit_Mix")
counts_df = results["counts"]
proportions_df = results["proportions"]
summary_df = results["summary"]


counts_df.index = counts_df.index.str.replace('Occupation_', '')
proportions_df.index = proportions_df.index.str.replace('Occupation_', '')
summary_df.index = summary_df.index.str.replace('Occupation_', '')

In [ ]:
custom_colors = {
    "prop_Bad": "orangered",
    "prop_Standard": "gold",
    "prop_Good": "skyblue"
}

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad", "prop_Standard", "prop_Good"],  
    stacked=False,                      
    title="Proporción de Credit Score por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=600,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()


fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Bad"],  
    stacked=False,                      
    title="Proporción de Bad por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Bad"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Good"],  
    stacked=False,                      
    title="Proporción Good por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Good"          
)

fig.show()

fig = plot_categorical_proportions(
    df=proportions_df.reset_index(),  
    x_column="index",                 
    y_columns=["prop_Standard"],  
    stacked=False,                      
    title="Proporción Standard por Ocupación",
    x_title="",
    y_title="Proporción",
    legend_title="Credit Score",
    colors=custom_colors,              
    width=1000,
    height=400,
    sort_order='asc',          
    sort_by="prop_Standard"          
)

fig.show()

# A simple vista no parece una buena variable:


<font color = 'skyblue'> Regresión

In [ ]:
# Regresión con variables dummy sobre todas las categorías:

# Se selecciona categoría base (la de menor probabilidad de default):

base_category = 'Occupation_Media_Manager'
y_col = "Credit_Score"

df_ = df.filter(regex = 'Occ').drop(base_category, axis = 1)

model = smf.ols(f"{y_col} ~ " + ' + '.join(df_.columns) + ' -1', data=df).fit()

print(model.summary())

# Todas las categorías tienen proporciones estadísticamente significativas y diferentes a la categoría base: 

                                 OLS Regression Results                                
Dep. Variable:           Credit_Score   R-squared (uncentered):                   0.631
Model:                            OLS   Adj. R-squared (uncentered):              0.630
Method:                 Least Squares   F-statistic:                          1.219e+04
Date:                Sun, 09 Mar 2025   Prob (F-statistic):                        0.00
Time:                        09:29:29   Log-Likelihood:                     -1.1787e+05
No. Observations:              100000   AIC:                                  2.358e+05
Df Residuals:                   99986   BIC:                                  2.359e+05
Df Model:                          14                                                  
Covariance Type:            nonrobust                                                  
                              coef    std err          t      P>|t|      [0.025      0.975]
----------------------------

In [40]:
y_column = "Credit_Score"

x_columns = [col for col in df.columns if "Occupation_" in col]

x_columns.remove("Occupation_Media_Manager") 

model = OrderedModel(df[y_column], df[x_columns], distr="logit")

result = model.fit(method='bfgs')

print(result.summary())

Optimization terminated successfully.
         Current function value: 1.060667
         Iterations: 30
         Function evaluations: 33
         Gradient evaluations: 33
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0607e+05
Model:                   OrderedModel   AIC:                         2.122e+05
Method:            Maximum Likelihood   BIC:                         2.123e+05
Date:                Sun, 09 Mar 2025                                         
Time:                        09:37:45                                         
No. Observations:              100000                                         
Df Residuals:                   99984                                         
Df Model:                          14                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------

In [49]:
insignificant_vars = result.pvalues[result.pvalues >= 0.05].index.tolist() + ['Occupation_Accountant']

# Excluir variables no significativas
significant_x_columns = [col for col in x_columns if col not in insignificant_vars]

# Ajustar nuevamente el modelo solo con las variables significativas
model_filtered = OrderedModel(df[y_column], df[significant_x_columns], distr="logit")
result_filtered = model_filtered.fit(method='bfgs')

# Mostrar los resultados
print(result_filtered.summary())


Optimization terminated successfully.
         Current function value: 1.060698
         Iterations: 19
         Function evaluations: 22
         Gradient evaluations: 22
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0607e+05
Model:                   OrderedModel   AIC:                         2.122e+05
Method:            Maximum Likelihood   BIC:                         2.123e+05
Date:                Sun, 09 Mar 2025                                         
Time:                        09:46:00                                         
No. Observations:              100000                                         
Df Residuals:                   99990                                         
Df Model:                           8                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------